# Phase 2 on real OmniDocBench data — InternVL2-2B

**Purpose.** Run the real adapter against the real OmniDocBench English-table subset (~14 pages). Unlike the previous notebook, this one feeds the model actual document pages with actual ground-truth tables. The output of this run is the first piece of *meaningful* model behavior the project produces.

**This notebook does NOT yet score quality.** Wiring the scorer into Phase 7 is the next step after this run succeeds. Here we only confirm: real adapter + real images + existing pipeline = sensible-looking output.

**Required runtime.** Colab Pro, GPU. ~5 GB free disk for model weights + a few hundred MB for the OmniDocBench subset.

Run cells top to bottom.

## 1. GPU sanity check

In [ ]:
import subprocess
try:
    out = subprocess.check_output(["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv,noheader"], text=True)
    print(out)
except Exception:
    print("CPU only — switch to GPU runtime before continuing.")

## 2. Clone the repo

In [ ]:
import os, subprocess
GIT_URL = "https://github.com/Michaelhamaty/Resarch_dev.git"
BRANCH = "main"
REPO_DIR = "/content/Research_claude"

if os.path.isdir(REPO_DIR):
    subprocess.check_call(["git", "-C", REPO_DIR, "fetch", "origin", BRANCH])
    subprocess.check_call(["git", "-C", REPO_DIR, "reset", "--hard", f"origin/{BRANCH}"])
else:
    subprocess.check_call(["git", "clone", "--branch", BRANCH, GIT_URL, REPO_DIR])
os.chdir(REPO_DIR)
print("HEAD:", subprocess.check_output(["git", "rev-parse", "--short", "HEAD"], text=True).strip())

## 3. Install dependencies

Same stack as the placeholder-fixture notebook, plus `huggingface_hub` for the dataset download.

In [ ]:
!pip install -q -e .
!pip install -q "transformers==4.40.0" "timm>=0.9" "einops>=0.7" "sentencepiece>=0.1.99" "accelerate>=0.27" "protobuf>=3.20" "huggingface_hub>=0.23"

## 4. Verify tests still pass

In [ ]:
!python -m pytest -q

## 5. Download the OmniDocBench subset

Pulls the annotations JSON, picks 14 English-table pages (deterministically, sorted by filename), downloads only those page images, and writes:

- `data/omnidocbench/images/<page_id>.<ext>`
- `data/omnidocbench/records.json`  (Phase 1 loader-compatible)
- `data/omnidocbench/ground_truth.json` (page_id → HTML — used by the scorer in a later step)

First run downloads ~50–200 MB depending on which pages get picked. Subsequent runs hit the HF cache and finish in seconds.

In [ ]:
!python scripts/data/build_omnidocbench_fixture.py --limit 14

In [ ]:
# Inspect the result.
import json, os
records = json.load(open("data/omnidocbench/records.json"))
gt = json.load(open("data/omnidocbench/ground_truth.json"))
print(f"records: {len(records)}    ground_truth entries: {len(gt)}")
print("\nfirst record:")
print(json.dumps(records[0], indent=2))
print("\nfirst GT (first 300 chars):")
first_id = records[0]["page_id"]
print(gt[first_id][:300])

## 6. Build Phase 1 manifests for the real subset

Same Phase 1 script, different config — points at `data/omnidocbench/records.json` and writes the splits under `data/splits/omnidocbench/`.

In [ ]:
!python scripts/subset_extraction/build_phase1_manifests.py --config configs/dataset/phase1_omnidocbench.yaml

## 7. Smoke-load the model + single-page chat() call

Same isolation pattern as the previous notebook: load the model, run on one real page, confirm output looks plausible before kicking off the full run.

In [ ]:
from PIL import Image
from adaptive_inference.config.budgets import load_budget
from adaptive_inference.config.models import load_model_config
from adaptive_inference.config.prompts import load_prompt_template
from adaptive_inference.inference.factory import build_adapter

cfg = load_model_config("configs/models/internvl2_real.yaml", "internvl2-2b")
adapter = build_adapter(cfg)
print("loaded:", adapter.device, adapter.dtype)

budget = load_budget("configs/budgets/phase2.yaml", "low")
prompt = load_prompt_template("configs/prompts/table_parse_v1.yaml")
rec = records[0]
img = Image.open(os.path.join("data", rec["image_path"])).convert("RGB")
print("image size:", img.size)

result = adapter.run(page_id=rec["page_id"], image=img, budget=budget, prompt=prompt)
print("\nruntime_ms:", round(result.runtime_ms, 1))
print("output_token_count:", result.output_token_count)
print("\n--- raw_text (first 2000 chars) ---")
print(result.raw_text[:2000])

## 8. Full Phase 2 run on the real calibration split (5 pages)

In [ ]:
!python scripts/main_runs/run_single_pass.py --config configs/runs/colab_real_2b_omnidocbench_low.yaml

## 9. Inspect the artifact tree and one page's output

In [ ]:
import os
RUN_DIR = "outputs/runs/colab_real_2b_omnidocbench_low_v1"
for root, dirs, files in os.walk(RUN_DIR):
    indent = "  " * (root.count(os.sep) - RUN_DIR.count(os.sep))
    print(f"{indent}{os.path.basename(root)}/")
    for f in sorted(files):
        print(f"{indent}  {f}  ({os.path.getsize(os.path.join(root, f))} B)")

In [ ]:
# Show the JSONL run log (per-page runtimes, token counts).
with open(f"{RUN_DIR}/run.log.jsonl") as f:
    for line in f:
        print(line.rstrip())

In [ ]:
# Show one page's raw model output, side-by-side with its ground truth.
import glob, json
raw_paths = sorted(glob.glob(f"{RUN_DIR}/raw/*.md"))
gt = json.load(open("data/omnidocbench/ground_truth.json"))
for p in raw_paths[:1]:
    page_id = os.path.basename(p).replace(".md", "")
    print("===", page_id, "===\n")
    print("--- MODEL OUTPUT (first 2000 chars) ---")
    with open(p) as f:
        print(f.read()[:2000])
    print("\n--- GROUND TRUTH (first 2000 chars) ---")
    print(gt.get(page_id, "(not found)")[:2000])

## 10. Download artifacts to your machine

In [ ]:
import shutil
ARCHIVE = "/content/colab_real_2b_omnidocbench_low_v1.zip"
shutil.make_archive(ARCHIVE.replace(".zip", ""), "zip", RUN_DIR)
print("Archive:", ARCHIVE)
try:
    from google.colab import files  # type: ignore
    files.download(ARCHIVE)
except Exception as e:
    print("Auto-download skipped (not in Colab UI):", e)

## Troubleshooting

| Symptom | Likely fix |
|---|---|
| `Could not locate OmniDocBench annotations JSON` | Inspect the dataset on HF: `https://huggingface.co/datasets/opendatalab/OmniDocBench`. Find the actual annotation filename and pass it via `--annotation-path PATH` to the build script. |
| `Could not find a list of page entries` | The annotations JSON has changed shape. Open it manually: `python -c "import json; print(list(json.load(open('/root/.cache/huggingface/...')))[:1])"` |
| `No English table pages found` | The selector got 0 matches. Either OmniDocBench changed its language values (e.g. `english` → `en`) or the table category name changed. The selector accepts both already; if you still hit this, post the first entry of the annotations JSON. |
| `404` on individual image download | The page entries reference an image path that does not exist as a separate file in the HF repo. Some OmniDocBench versions ship images in a tarball — adjust the script to use `snapshot_download` for the images dir. |
| All other errors (model load, GPU, etc.) | See the troubleshooting section in `notebooks/colab_phase2_real_2b.ipynb` — they all apply here too. |